# Tutorial -  Volatilidad
Sergio Cabrales, Universidad de los Andes

https://www.sac2.com/

## 1. Carga de librerías, funciones y APIs necesarias.

#### 1.1. Instalan las librerías que no incluye Google Colab

In [ ]:
pip install yfinance

In [ ]:
pip install mplfinance

In [ ]:
pip install arch

#### 1.2. Se cargan las librerías requeridas

In [ ]:
# Funciones numéricas adicionales
import numpy as np

# Lectura de datos y manejo de Data-sets
import pandas as pd

# Datos
import yfinance as yfin

# Gráficos
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
import seaborn as sns

#analisis tecnico
import mplfinance as mpf

# Probabilidad y estadística
import math
from scipy.stats import norm, chi2, jarque_bera,shapiro
from scipy.optimize import brentq
import statsmodels.api as sm
from scipy import stats

# ARCH model
from arch import arch_model

# Engel test for ARCH effects
from statsmodels.stats.diagnostic import het_arch

## 2. Obtención de datos históricos

#### 2.1. Descarga de datos desde Yahoo Finance

https://finance.yahoo.com/


In [ ]:
# Descargamos datos de la acción sleccionada:
df = yfin.download('^GSPC', start='2020-01-01', multi_level_index=False)
df = df.dropna()
df

In [ ]:
df.to_excel('df.xlsx', index=False)

## 3. Visualización y Estadísticas Descriptivas

### 3.1. Utiliza la librería mpf para hacer un gráfico de velas japonesas de la acción

In [ ]:
mpf.plot(df,type='candle', volume=True,figratio=(19,8),style='yahoo',title='S&P500')

## 4. Retornos

### 4.1. Retornos Logarítmicos

Los retornos logarítmicos se calculan como:
$$
r_{t} = ln \left( \frac{S_t}{S_{t-1}} \right ) = ln \left( S_{t} \right) - ln \left( S_{t-1} \right)
$$

In [ ]:
# Guardamos los retornos logaritmicos en una nueva columna.
df['Log Returns'] = np.log(df['Close']) - np.log(df['Close'].shift(1))
df['Log Returns'][0] = 0
df

### 4.3. Retornos Logarítmicos anualizados

Podemos calcular el log-retorno anual ($r$) como el número de días bursátiles (252 días) por el promedio del log-retorno diario:

$$
r = 252 \bar{r_t}
$$

In [ ]:
# Podemos imprimir el retornos anual:
LogReturns = np.mean(df["Log Returns"])*252
LogReturns

### 4.4. Gráfica de retornos
- Podemos graficar los retornos igual que como graficamos los precios.

In [ ]:
# Gráfico de los retornos logarítmicos
plt.figure(figsize=(15,8))
plt.plot(df['Log Returns'], color = 'red')
plt.title('Retornos Logarítmicos')
plt.xlabel('Fecha')
plt.show()

## 5. Volatilidad

### 5.1 Volatilidad diaria y anual

La volatilidad diaria del activo es la desviación estándar de sus retornos o la raíz de la varianza:

$$vol=desv(r)=\sqrt{Var(r)}$$

En finanzas, se utiliza con mayor frecuencia la volatilidad anualizada ($\sigma$) en lugar de la volatilidad diaria. Teniendo en cuenta que en cada año hay 252 días bursátiles:

$$ \sigma^{2} = \sum_{1}^{252} Var_{diaria}$$
$$ \sigma^{2} = 252 \sigma_{diaria}^{2}$$

Se saca la raíz cuadra a ambos lados para calcular la volatilidad:

$$ \sqrt{\sigma^{2}} = \sqrt{252 \sigma_{diaria}^{2}}$$
$$ \sigma = \sigma_{diaria} \sqrt{252}$$

In [ ]:
# Calculamos la volatilidad diaria con los retornos logaritmicos.
vol_d = np.std(df['Log Returns'])

# Anualizamos la volatilidad diaria.
vol_a = vol_d * np.sqrt(252)

print("Volatilidad diaria: {:.4f} %".format(100*vol_d))
print("Volatilidad anualizada: {:.4f} %".format(100*vol_a))

## 6. Normalidad de los retornos

### 6.1. Histogram

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(df['Log Returns'])))

plt.hist(df['Log Returns'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Log-Returns')

# Show the plot
plt.show()

### 6.2. Q-Q Plot

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
sm.qqplot(df['Log Returns'], stats.norm, loc=mean, scale=std_dev, line='45')
# Se agrega un título a la gráfica
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 6.3. P-P Plot

In [ ]:
# Se crea una PP plot calculando las probabilidades empíricas y teóricas
mean = np.mean(df['Log Returns'])
std_dev = np.std(df['Log Returns'])
n = len(df['Log Returns'])
fig, ax = plt.subplots()
# Se calculan las probabilidades empíricas
p = np.arange(1, n + 1) / n - 0.5 / n
# Se calculan las probabilidades teóricas
pp = np.sort(stats.norm.cdf(df['Log Returns'],loc=mean, scale=std_dev ))
sns.scatterplot(x=pp, y=p, color='blue', edgecolor='blue', ax=ax)
ax.set_title('P-P plot')
ax.set_xlabel('Theoretical Probabilities')
ax.set_ylabel('Sample Probabilities')
ax.margins(x=0, y=0)
# Se dibuja la línea roja de 45°
plt.plot(np.linspace(0, 1.01), np.linspace(0, 1.01), 'r', lw=2)
# Se muestra la gráfica
plt.show()

### 6.4. Jarque-Bera test

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(df['Log Returns'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

### 6.5. Shapiro-Wilk test

In [ ]:
# perform Shapiro-Wilk test for normality
shapiro_test = shapiro(df['Log Returns'])
print('Shapiro-Wilk test p-value: ', shapiro_test[1])

## 7. Autocorrelación

### 7.1. Autocorrelation

In [ ]:
# Crear el autocorrelograma
plot_acf(df['Log Returns'], lags = 19)
plt.show()

In [ ]:
df['Log Returns^2'] = df['Log Returns']**2

In [ ]:
# Crear el autocorrelograma de retornos al cuadrado
plot_acf(df['Log Returns^2'], lags = 42)
plt.show()

### 7.2. Partial Autocorrelation

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns'], lags = 19)
plt.show()

In [ ]:
# Crear el autocorrelograma parcial
plot_pacf(df['Log Returns^2'], lags = 19)
plt.show()

### 7.3. Ljung-Box test

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 19
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

## 8. ARCH effects

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 19
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(df['Log Returns^2'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

### 9.1. ARCH model

$ \sigma_{t}^{2} = \alpha_{0} + \sum^{q}_{j=1} \alpha_j ϵ_{t-j}^{2}  $

In [ ]:
# Create the ARCH model
model = arch_model(df['Log Returns'], mean='Zero', vol='ARCH', p=19, q=0, rescale=False)

# Fit the model
results = model.fit()

# Print the results summary
print(results.summary())

In [ ]:
params = pd.DataFrame(results.params)
params

In [ ]:
# get residuals
e = pd.DataFrame(results.std_resid)
e['resid^2'] = e['std_resid']**2
e

### 9.2. Histogram of Residuals

In [ ]:
# Create a histogram with k bins
k = int(math.sqrt(len(e['std_resid'])))

plt.hist(e['std_resid'], bins=k)

# Add labels and a title
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals')

# Show the plot
plt.show()

### 9.3. Q-Q plot - Residuals

In [ ]:
# Se crea una Q-Q plot con el paquete statsmodels.api
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.qqplot(line="45")
plt.title("Q-Q Plot")
# Se muestra la gráfica
plt.show()

### 9.4. P-P plot - Residuals

In [ ]:
# Se crea una P-PQ plot
pplot = sm.ProbPlot(e['std_resid'], stats.norm, fit=True)
# Se agrega un título a la gráfica
fig = pplot.ppplot(line="45")
plt.title("P-P Plot")
# Se muestra la gráfica
plt.show()

### 9.5. Autocorrelation - Residuals

In [ ]:
# Crear el autocorrelograma
plot_acf(e['std_resid'], lags = 19)
plt.show()

In [ ]:
# Crear el autocorrelograma
plot_acf(e['resid^2'], lags = 19)
plt.show()

### 9.6. Jarque Bera Test - Residuals

In [ ]:
# perform the Jarque-Bera test
jb_value, p_value = jarque_bera(e['std_resid'])

# print the results
print("Jarque-Bera value: ", jb_value)
print("p-value: ", p_value)

if p_value > 0.05:
    print("The data is normally distributed")
else:
    print("The data is not normally distributed")

### 9.7. Efecto ARCH

In [ ]:
# Define the number of lags for the test (usually chosen based on data characteristics)
lags = 19
# perform the Ljung-Box test
lb_test = sm.stats.diagnostic.acorr_ljungbox(e['resid^2'], lags)

# print the results
print("Ljung-Box value: ", lb_test.iloc[-1, 0])
print("p-value: ", lb_test.iloc[-1, 1])

if lb_test.iloc[-1, 1] > 0.05:
    print("The data is independent")
else:
    print("The data is not independent")

### 9.8. Volatility - Forecast

In [ ]:
df['Volatility'] = results.conditional_volatility*np.sqrt(252)
fig = results.plot(annualize="D")

In [ ]:
# Gráfico de la volatilidad
plt.figure(figsize=(15,8))
plt.plot(df['Volatility'], color = 'red')
plt.title('Volatility - ARCH')
#plt.xlabel('Fecha')
plt.ylabel('Volatility')
plt.show()

In [ ]:
print("El último valor de volatilidad es", round(df['Volatility'][-1]*100,4),"%")